In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import seaborn as sns
import scipy as sp
from sklearn import linear_model
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

pa_gs = pd.read_csv('data/pa_data.csv').dropna()

In [19]:
X_pca = pa_gs[['dtr_annual', 'dtr_spring', 'dtr_summer', 'tmax_annual', 'prcp_annual_mm', 'prcp_growing_season_mm', 
    'prcp_spring_mm', 'latitude', 'longitude', 'elevation_m', 'dist_coast_km', 'dist_greatlakes_km', 'dist_atlantic_km',
    'oni_annual', 'nao_annual', 'nao_djf', 'pna_annual', 'amo_annual', 'ohc700_atlantic', 'ohc700_atlantic_se', 
    'ohc700_north_atlantic', 'ohc700_north_atlantic_se', 'ohc700_south_atlantic', 'ohc700_south_atlantic_se', 
    'ohc2000_north_atlantic', 'ohc700_pacific', 'ohc700_world','ohc700_natl_djf', 'ohc700_natl_amj', 'sst_north_atlantic',
    'sst_gulf_stream', 'sst_gulf_mexico', 'sst_tropical_north_atl', 'pwat_eastern_us', 'pwat_southeast_us', 
    'pwat_northeast_us', 'pwat_gulf_coast', 'pwat_station', 'dewpoint_2m_eastern_us', 'soil_moisture_eastern_us',
    'cloud_cover_eastern_us', 'evaporation_eastern_us', 'dewpoint_2m_southeast_us', 'soil_moisture_southeast_us',
    'cloud_cover_southeast_us', 'evaporation_southeast_us', 'dewpoint_2m_northeast_us', 'soil_moisture_northeast_us',
    'cloud_cover_northeast_us', 'evaporation_northeast_us', 'dewpoint_2m_pennsylvania', 'soil_moisture_pennsylvania',
    'cloud_cover_pennsylvania', 'evaporation_pennsylvania', 'dewpoint_station', 'soil_moisture_station',
    'cloud_cover_station', 'evaporation_station']]
y_pca = pa_gs['growing_season_length']
y_reg = pa_gs['growing_season_length']

In [16]:
def principal_component_analysis(X, y):
    #creates and graphs principal component analysis for inputted X and y
    #required libraries: from sklearn.preprocessing import StandardScalar, from sklearn.decomposition import PCA,
    #from sklearn.linear_model import LinearRegression, matplotlib.pyplot as plt, numpy as np, seaborn as sns

    #gets list of columns in input X for future conversion of X_train back into dataframe
    columns_X = list(X.columns)

    # splits data into X_train, X_test, y_train, and y_test and makes X_train a dataframe
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 0)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_train = pd.DataFrame(X_train, columns = columns_X)

    # First PCA
    pca1 = PCA()
    X_pca1 = pca1.fit_transform(X_train)
    #plt.bar(range(1,len(pca1.explained_variance_)+1),pca1.explained_variance_)
    #plt.ylabel('Explained Variance')
    #plt.xlabel('Components')
    #plt.plot(range(1,len(pca1.explained_variance_)+1), np.cumsum(pca1.explained_variance_), 
    #     c='red', label = 'Cumulative Explained Variance')
    #plt.legend(loc='upper left')
    #plt.show()

    # Second PCA
    pca2b = PCA(n_components=2)
    X_pca2b = pca2b.fit_transform(X_train)
    #colormap = plt.get_cmap('coolwarm')
    #plt.figure()
    #scatter = plt.scatter(X_pca2b[:, 0], X_pca2b[:, 1], c=y_train, cmap= colormap)
    #plt.xlabel('PC1')
    #plt.ylabel('PC2')
    #plt.colorbar(scatter, label = 'Growing Season Length')
    #plt.show()

    # Third PCA
    pca3 = PCA(n_components=3)
    X_pca3 = pca3.fit_transform(X_train)

    #fig = plt.figure()
    #ax = fig.add_subplot(111, projection='3d')
    #ax.scatter(X_pca3[:, 0], X_pca3[:, 1], X_pca3[:, 2], c=y_train, cmap=colormap)
    #ax.set_xlabel('PC1')
    #ax.set_ylabel('PC2')
    #ax.set_zlabel('PC3')
    #plt.show()

    loadings = pca3.components_.T * np.sqrt(pca3.explained_variance_)
    #plt.figure(figsize=(10, 8))
    #sns.heatmap(loadings, annot=True, cmap='coolwarm', xticklabels=['PC1', 'PC2', 'PC3'], yticklabels=X.columns)
    #plt.title('Feature Importance in Principal Components')
    #plt.show()

    return [X_pca1[:, 0], X_pca1[:, 1], X_pca1[:, 2], X_pca1[:, 3], X_pca1[:, 4]]

def multiple_regression(input_variables, predicted_variable):
    #OLD VERSION W/OUT DROPPING NaN VALUES AUTOMATICALLY
    #performs a standardized multiple linear regression to create a model of y based on the variables within X.
    # Required Libraries: r2score, mean_absolute_error, and mean_squared_error from sklearn.metrics, 
    # train_test_split from sklearn.model_selection, linear_model from sklearn, and StandardScaler from sklearn.preprocessing
    
    tot_data_points = int(len(input_variables))
    
    #splits data into training and testing groups
    X_train, X_test, y_train, y_test = train_test_split(input_variables, predicted_variable, test_size = 0.2, random_state =0)

    #scales input data to standardize to mean of 0 and standard deviation of 1
    sc = StandardScaler()
    X_train_scaled = sc.fit_transform(X_train)
    X_test_scaled = sc.fit_transform(X_test)
    
    #creates multiple regression model based on training data
    regr = linear_model.LinearRegression()
    regr.fit(X_train_scaled, y_train)

    #calculates mean absolute error, mean squared error, and r squared score based on the predicted y
    y_predicted = regr.predict(X_test_scaled)
    mae = mean_absolute_error(y_test, y_predicted)
    mse = mean_squared_error(y_test, y_predicted)
    rmse = np.sqrt(mse)
    r_2_score = r2_score(y_test, y_predicted)
    n = len(predicted_variable)
    p = input_variables.shape[1]
    adj_r2_score = 1 - ((1 - r_2_score) * (n - 1) / (n - p - 1))
    y_std = predicted_variable.std()
    coefficients = pd.DataFrame(zip(input_variables.columns, input_variables.std(), regr.coef_))

    coefficients.columns = ['variable', 'var standard dev', 'coefficient']
    
    results = [f'MAE = {mae}', f'MSE = {mse}', f'RMSE = {rmse}', f'R Squared = {r_2_score}', 
               f'Adj. R Squared = {adj_r2_score}', f'Total Data Points: {tot_data_points}', 
               f'(Reference) predicted variable standard deviation = {y_std}', coefficients]
    
    return display(results)

In [28]:
PC1, PC2, PC3, PC4, PC5 = principal_component_analysis(X_pca, y_pca)
X_reg = {'PC1':PC1, 'PC2':PC2, 'PC3':PC3, 'PC4':PC4, 'PC5':PC5}
X_reg = pd.DataFrame(X_reg)
print(X_reg)

           PC1       PC2       PC3       PC4       PC5
0     4.200903  2.266149 -1.251272  0.257746 -1.061950
1    -5.364875 -0.356090 -2.776940  1.809321  0.150180
2     0.424776  3.102663 -3.610300 -1.555081 -0.760818
3     5.225733  1.468391  1.389804 -3.436891 -0.300651
4    -3.169797 -3.650398 -1.901492  1.592043  0.778671
...        ...       ...       ...       ...       ...
1387  5.100438 -1.397543  2.317601  1.967863  1.600728
1388 -2.543011  0.690905  2.337841  1.011525 -2.635643
1389  7.116086 -5.222837  1.992528 -1.070207  0.291402
1390 -2.667325 -5.364328 -2.149780 -0.604127  3.064102
1391 -2.035390 -4.765587  3.520186  2.563940  2.405983

[1392 rows x 5 columns]


In [32]:
multiple_regression(X_reg, y_reg)

KeyError: 'key of type tuple not found and not a MultiIndex'